# 🧹 Germany Housing Market — Phase 2: Clean the Data

## 🎯 Goal of this phase
Turn the **raw** file (`data/raw/immo_data.csv`) into a **clean** dataset we can analyze. We will remove duplicates, drop impossible values, handle missing data, and create useful new columns — then save the result to `data/processed/immo_data_clean.csv`.

## 📖 How to use this notebook
Same rules as Phase 1: read the Markdown, then **write the code yourself** in each cell (replacing the `# TODO`), run it, and compare with the **Expected output** note.

> 💡 Phase 1 taught you what is wrong with the data. This phase fixes it. If you did not finish Phase 1, go back and do it first — your findings will tell you what to clean.

## 📥 Carried over from before (your open items)

Phase 1 is closed as *exploration* — nothing below has been acted on yet. These findings are its output and the **input** to the cleaning below. Every number was verified against `data/raw/immo_data.csv`: Phase 1 numbers on 2026-09-17, the Step 2 duplicate check on 2026-09-24.

### A. Decisions this notebook has to make

| Phase 1 finding | Where it gets decided |
|---|---|
| **No duplicate rows** — `scoutId` is unique (268,850 unique = 268,850 rows, 0 nulls); full-row duplicates also 0 | Step 2 is a **no-op**: keep the check, drop nothing → **Step 2** |
| `regio1` == `geo_bln` (identical, 16 states, 0 nulls) | drop one of the pair → **Step 4** |
| `regio2` == `geo_krs` (identical, 419 districts, 0 nulls) | drop one of the pair → **Step 4** |
| `regio2` is a **district**, not a city | build `city` from **`regio3`** instead → **Step 5** |
| `date` holds only **4** snapshots: `Sep18`, `May19`, `Oct19`, `Feb20`, unequal gaps, stops Feb20 | parse in **Step 5**; the caveat must survive into Phase 3 |
| 7 columns are > 60 % empty | drop list → **Step 4** |
| `totalRent` is 15.07 % empty (≈ 40,500 rows) vs `baseRent` 0 % | decide: fill, drop, or analyse `baseRent` as the primary price → **Step 4 / 7** |

### A2. Checked, no action needed

> 🔁 **Look-alike rows — checked, no action.** Grouping on `baseRent + livingSpace + noRooms + yearConstructed + geo_plz + regio3 + date` matches 15,519 rows in 6,892 clusters. These are **most likely** distinct units rather than duplicate records — each has its own `scoutId`, and `picturecount` often differs (largest cluster: 31 flats, 31 different ids, 3 vs 4 photos) — though no column in the dataset *proves* it, since one flat could in principle be re-listed under a new id. Either way the impact is negligible: **0** of the 1,247 places with ≥ 50 listings have any single signature above 20 % of that place (worst among places with ≥ 100 listings: Delitzsch, 31 / 231 = 13.4 %). Nothing to remove and not worth a Phase 3 caveat.

> 🔤 **Text encoding is clean.** The file is UTF-8 with no BOM and umlauts are genuinely present (`Schüren`, `Bühl_Iggelheim`). If you ever see `�` in terminal output it is console rendering, not corrupted data — so Step 5 only needs `.str.replace('_', ' ')`, no umlaut repair.

### B. Verified extremes — use these as your Step 3 bounds

| Column | min | max | note |
|---|---|---|---|
| `baseRent` | 0 | 9,999,999 | std **19,536** vs mean **694** → a handful of rows would dominate every average |
| `totalRent` | 0 | 15,751,535 | 15.07 % missing |
| `livingSpace` | 0 | 111,111 | m² |
| `noRooms` | 1 | 999.99 | |

### C. Verified list of columns above 60 % missing (Step 4 drop list)

`telekomHybridUploadSpeed` (83.25 %), `electricityBasePrice` (82.58 %), `electricityKwhPrice` (82.58 %), `energyEfficiencyClass` (71.07 %), `lastRefurbish` (69.98 %), `heatingCosts` (68.19 %), `noParkSpaces` (65.39 %).

### D. Text style (Step 5)

Names use `_` instead of spaces: `Nordrhein_Westfalen`, `Äußere_Neustadt_Antonstadt`, `Südliche_Weinstraße_Kreis`. Tableau will show these verbatim until you clean them.

### E. Stays behind in Notebook 1

- the clean `Run All` state — notebook 1 now executes 40 → 48 in order. Keep it that way: a phase is only "done" when it reproduces.

> 🚨 **Scope risk to carry forward:** with only 4 irregular time points, plan questions 6, 8 and 9 ("rising fastest", "price appreciation", "ROI trend") cannot be answered credibly. Decide in Phase 3 whether to reframe them as *snapshot comparisons* instead — for example "how did rent per m² in the top 10 cities move between Sep18 and Feb20" rather than a continuous trend.


## Step 1 — Setup: imports + reload the raw data

**What & why:** Notebooks do not remember variables from other notebooks, so we start fresh: import the tools, then load the raw CSV again into a new `df`.

**Input:** `../data/raw/immo_data.csv`.

**Hints (not the answer):**
- Import `pandas as pd` and `numpy as np` (as in Phase 1).
- Read the CSV with `pd.read_csv(...)` into `df`, then print `df.shape`.
- Keep the raw data untouched — we only read it. All cleaning happens on a copy we build step by step.
- Set the float display format as in Phase 1 (`pd.set_option(...)`) so large numbers stay readable.

**Expected output (verified):** exactly `(268850, 49)` — the same shape Phase 1 saw. If your column count differs, stop and investigate *before* cleaning anything.

In [7]:
# TODO 1 — import pandas and numpy, load the raw CSV into df, print df.shape
import pandas as pd
import numpy as np
pd.set_option("display.float_format", "{:,.2f}".format)
df= pd.read_csv("../data/raw/immo_data.csv")
print(df.shape)

(268850, 49)


## Step 2 — Check for duplicate listings

**What & why:** The same apartment may appear more than once. Each listing has a unique id column named `scoutId` — rows with the same `scoutId` would be duplicates, so that is the column to check.

**Input:** `df`.

**Hints (not the answer):**
- Count duplicates first: `df.duplicated(subset=['scoutId']).sum()`.
- Then remove them with `df.drop_duplicates(subset=['scoutId'], keep='first')`.
- Save the result back into `df` and print the number of rows **before vs after**.

**Expected output (verified 2026-09-24):** a duplicate count of **0**, and **268,850 rows before → 268,850 rows after**. The drop removes nothing.

> ✅ That is the correct answer, not a bug. `scoutId` is `int64`, has 0 nulls, and holds **268,850 unique values across 268,850 rows**, so no two listings share an id. Run the `drop_duplicates` anyway: it costs nothing, it documents the check for anyone reading your pipeline, and it would catch duplicates if you ever re-downloaded a newer version of the dataset.

> 🧠 **Why keep a step that does nothing?** A cleaning step is only pointless if its answer *cannot* change the data. This one can — if duplicate `scoutId`s existed, deleting them would be a real fix. The check is conditional, and today the condition happens to be false. Steps like that are normal in cleaning; keep them.


In [8]:
# TODO 2 — count duplicates on scoutId, drop them, report rows before vs after
print(f"There are {df.duplicated(subset=['scoutId']).sum()} duplicate rows, and there are currently {df.shape[0]} rows in the table")
df= df.drop_duplicates(subset= ['scoutId'], keep='first')
print(f"After removing duplicate rows, there are currently {df.shape[0]} rows in the table")


There are 0 duplicate rows, and there are currently 268850 rows in the table
After removing duplicate rows, there are currently 268850 rows in the table


## Step 3 — Remove impossible values

**What & why:** Some rows are clearly data errors (a 1 m² flat, a €0 rent). Your Phase 1 numbers show how extreme they are: `baseRent` runs from 0 to 9,999,999 with a standard deviation of 19,536 against a mean of 694 — meaning a handful of rows would otherwise dominate every average you compute later. Removing them is not optional.

**Input:** `df` (see the carried-over table in section B above).

**Structure — three moves, in order:**

- **3a — logical contradictions:** drop rows that cannot be true: `baseRent > totalRent` (when totalRent ≠ 0), both rents = 0, `noRooms = 0`, `livingSpace = 0`.
- **3b — investigation:** measure the rent mismatches (`totalRent ≠ baseRent + heatingCosts + serviceCharge`) and look at the biggest outliers. Exploration only — the keep/drop decision is recorded at the end of the step.
- **3c — sane bounds:** drop rows outside fixed limits: `baseRent` 10–10,000 €, `livingSpace` 10–500 m², `noRooms` 1–20, plus a `totalRent` cap at 20,000 € (NaN and 0 stay for Step 4 to decide).

**Hints (not the answer):**
- Build **boolean masks** and combine them with `&` — every comparison needs its own parentheses: `(a == b) & (c != d)`.
- Keep the rows that pass **all** checks, then report how many rows were removed.

**Decision (made 2026-11-03):** fixed bounds, **not** 99th-percentile clipping. The 99th percentiles are baseRent €2,500 / livingSpace 180 m² — those are *real* expensive flats (Munich!), and cutting the top 1% would bias every city comparison. Fixed bounds only remove provably broken rows; skew is handled in Phase 3 with medians and log scales.

**Expected output (verified against the raw file):**

| Check | Rows removed |
|---|---|
| c1 — baseRent > totalRent (totalRent ≠ 0) | 215 |
| c2 — both rents = 0 | 7 |
| c3 — noRooms = 0 | 0 |
| c4 — livingSpace = 0 | 72 |
| **3a subtotal** | **294** |
| bounds (rent / space / rooms) | 216 |
| totalRent cap (> 20,000 €) | 8 |
| **Step 3 total** | **518 (0.19 %)** |

→ **268,332 rows survive (99.81 % of the original).**

> ⚠️ Your mismatch counts will shift slightly versus your first run (35,340 → 35,336): conditions 3 + 4 now run *before* the investigation, so a few broken rows no longer pollute the mismatch count.

In [8]:
#Creating a clean copy of the dataframe where cleaning will take place
# removing rows based on logical contradictions and according to count
    #condition 1: baseRent>totalRent and totalRent!=0 (As zero could simply be a missed entry rather than a faulty one)
    #condition 2: totalRent = 0 and baseRent = 0
    #condition 3: noRooms = 0
    #condition 4: LivingSpace = 0

#CONDITION 1
df_clean = df.copy()

mask1 = (df_clean['baseRent'] > df_clean['totalRent']) & df_clean['totalRent']!=0
df_clean = df_clean.drop(df_clean[mask1].index)

#CONDITION 2
mask2 = (df_clean['baseRent']==0) & (df_clean['totalRent']==0)
df_clean = df_clean.drop(df_clean[mask2].index)

#CONDITION 3 
# Investigating rows where heating costs and service charg are available 
# (Rows where base and total rent are not matching)
rent_mismatch_df = df_clean.loc[(df_clean['heatingCosts'].notna()) & (df_clean['serviceCharge'].notna()) & (df_clean['totalRent']!= df_clean['baseRent']+ df_clean['heatingCosts']+ df_clean['serviceCharge']) ,['baseRent', 'totalRent', 'heatingCosts', 'serviceCharge','electricityBasePrice','noRooms', 'livingSpace']]
print(f"There are {rent_mismatch_df.shape[0]} rows with mismatching total and base rents")
total_rent_available_df = rent_mismatch_df[rent_mismatch_df['totalRent'].notna()]
total_rent_missing_df =rent_mismatch_df[~rent_mismatch_df['totalRent'].notna()]
print(f"Of these {rent_mismatch_df.shape[0]} rows, {total_rent_available_df.shape[0]} have an actual mismatch, and {total_rent_missing_df.shape[0]} have a missing total rent")


There are 35340 rows with mismatching total and base rents
Of these 35340 rows, 23058 have an actual mismatch, and 12282 have a missing total rent


In [9]:
# --- 3a: all four logical-contradiction drops, rebuilt fresh from df ---
# what: rebuild df_clean from the untouched df so this cell is re-runnable on its own
# why: the cell above (your first attempt) is superseded by this one — you can delete it
df_clean = df.copy()
rows_before = df_clean.shape[0]

# what: mask 1 = total rent SMALLER than base rent (total = base + extras, so this is impossible)
# why: NaN totalRent must survive (missing ≠ contradictory — NaN comparisons return False automatically), hence the explicit != 0
mask1 = (df_clean['baseRent'] > df_clean['totalRent']) & (df_clean['totalRent'] != 0)   # FIXED: parentheses added — the old version only worked by luck of operator precedence
n1 = int(mask1.sum())
df_clean = df_clean[~mask1]

# what: mask 2 = BOTH rents are 0 — no rent information at all
# why: a listing with zero base rent AND zero total rent is an ad without a price
mask2 = (df_clean['baseRent'] == 0) & (df_clean['totalRent'] == 0)
n2 = int(mask2.sum())
df_clean = df_clean[~mask2]

# what: mask 3 = zero rooms
# why: verified earlier this matches 0 rows — we keep the check anyway so it guards future data reloads
mask3 = df_clean['noRooms'] == 0
n3 = int(mask3.sum())
df_clean = df_clean[~mask3]

# what: mask 4 = zero living space
# why: a 0 m² flat does not exist — these 72 rows are data errors
mask4 = df_clean['livingSpace'] == 0
n4 = int(mask4.sum())
df_clean = df_clean[~mask4]

# what: report each condition's removal count so the cleaning is measurable, not silent
# why: the Step 3 markdown table cites exactly these numbers
print(f"c1 baseRent>totalRent: {n1} | c2 both=0: {n2} | c3 noRooms=0: {n3} | c4 space=0: {n4}")
print(f"3a removed {rows_before - df_clean.shape[0]} rows -> {df_clean.shape[0]} remaining")

# --- rebuild the mismatch investigation frames (3b) ---
# what: rows where totalRent != baseRent + heatingCosts + serviceCharge (both extras filled)
# why: same investigation you wrote above, now run AFTER conditions 1-4 so broken rows don't pollute the count
rent_mismatch_df = df_clean.loc[
    df_clean['heatingCosts'].notna()
    & df_clean['serviceCharge'].notna()
    & (df_clean['totalRent'] != df_clean['baseRent'] + df_clean['heatingCosts'] + df_clean['serviceCharge']),
    ['baseRent', 'totalRent', 'heatingCosts', 'serviceCharge', 'noRooms', 'livingSpace'],
]

# what: split the mismatches into "totalRent filled" vs "totalRent missing"
# why: a missing totalRent is a Step 4 problem, while a filled-but-wrong one is a data-quality observation
total_rent_available_df = rent_mismatch_df[rent_mismatch_df['totalRent'].notna()]
total_rent_missing_df = rent_mismatch_df[rent_mismatch_df['totalRent'].isna()]

# what: print the split so the numbers land in the notebook output
# why: 23,057 / 12,279 are the verified counts the 3b markdown cites
print(f"mismatching total vs base+extras: {rent_mismatch_df.shape[0]} "
      f"(filled: {total_rent_available_df.shape[0]} | missing: {total_rent_missing_df.shape[0]})")


c1 baseRent>totalRent: 215 | c2 both=0: 7 | c3 noRooms=0: 0 | c4 space=0: 72
3a removed 294 rows -> 268556 remaining
mismatching total vs base+extras: 35336 (filled: 23057 | missing: 12279)


In [6]:
#Investigating Mismatching total and base rents where the total rent is available
total_rent_available_df[total_rent_available_df['totalRent']!=total_rent_available_df['baseRent']+total_rent_available_df['serviceCharge']]



,baseRent,totalRent,heatingCosts,serviceCharge,noRooms,livingSpace
14,972.60,"1,320.65",43.05,215.00,3.00,87.00
19,"1,343.48","1,878.48",160.00,290.00,5.00,127.95
38,400.00,570.00,75.00,150.00,3.50,70.00
68,640.00,960.00,125.00,155.00,3.00,82.00
95,284.09,466.72,81.17,101.46,4.00,67.64
...,...,...,...,...,...,...
268704,400.00,450.00,0.00,30.00,3.00,76.00
268715,525.00,675.00,65.00,60.00,2.00,50.03
268824,"1,044.00","1,298.00",102.00,255.00,3.00,101.81
268826,309.61,428.69,59.54,59.54,3.00,59.54


In [4]:
# TODO 3 — filter out impossible values using boolean masks; report rows removed
display(df[['baseRent', 'totalRent', 'noRooms', 'livingSpace']].describe())
print(f"Below is the 99th percentile of the columns being investigated for outliers\n{df[['baseRent', 'totalRent', 'noRooms', 'livingSpace']].quantile(0.99)}")
display(df[['baseRent', 'totalRent', 'serviceCharge', 'heatingCosts', 'noRooms', 'livingSpace']].sort_values('baseRent', ascending=False).head(10))
display(df[['totalRent', 'baseRent', 'serviceCharge', 'heatingCosts', 'noRooms', 'livingSpace']].sort_values('totalRent', ascending=False).head(10))

,baseRent,totalRent,noRooms,livingSpace
count,"268,850.00","228,333.00","268,850.00","268,850.00"
mean,694.13,901.33,2.64,74.36
std,"19,536.02","33,238.33",2.63,254.76
min,0.00,0.00,1.00,0.00
25%,338.00,469.80,2.00,54.00
50%,490.00,650.00,3.00,67.32
75%,799.00,985.00,3.00,87.00
max,"9,999,999.00","15,751,535.00",999.99,"111,111.00"


Below is the 99th percentile of the columns being investigated for outliers
baseRent      2,500.00
totalRent     2,850.00
noRooms           5.00
livingSpace     180.00
Name: 0.99, dtype: float64


,baseRent,totalRent,serviceCharge,heatingCosts,noRooms,livingSpace
264934,"9,999,999.00",NaN,NaN,NaN,1.00,1.00
8406,"1,234,567.00","1,234,567.00",NaN,NaN,4.00,76.00
260259,"1,000,000.00","1,000,000.00",NaN,NaN,1.00,10.00
16731,"120,000.00",NaN,NaN,NaN,3.00,65.72
243713,"120,000.00",NaN,"25,000.00",NaN,3.50,"14,000.00"
181618,"39,200.00",NaN,165.00,NaN,1.00,49.00
176647,"30,990.00",449.90,78.00,62.00,3.00,47.67
213625,"20,100.00","2,390.00",140.00,150.00,3.00,110.00
190283,"20,000.00","26,500.00","6,500.00",NaN,10.00,566.00
170816,"19,329.00","20,588.00","1,259.00",NaN,6.00,601.85


,totalRent,baseRent,serviceCharge,heatingCosts,noRooms,livingSpace
239533,"15,751,535.00","1,325.00",250.00,NaN,3.00,125.00
8406,"1,234,567.00","1,234,567.00",NaN,NaN,4.00,76.00
109047,"1,150,900.00",950.00,200.00,NaN,5.00,137.00
260259,"1,000,000.00","1,000,000.00",NaN,NaN,1.00,10.00
79903,"485,350.00",350.00,135.00,NaN,3.50,62.00
23490,"108,000.00",820.00,260.00,NaN,2.50,86.61
197378,"64,651.00",487.51,159.00,NaN,3.00,79.00
16832,"63,204.00",460.29,171.75,NaN,2.00,68.70
97927,"51,570.00",427.70,50.00,38.00,1.00,32.90
240954,"37,600.00",266.00,110.00,NaN,2.00,49.00


### 3b — Investigate the rent mismatches (exploration, no dropping yet)

`totalRent` should equal `baseRent + heatingCosts + serviceCharge`. Because 3a now runs first, these counts are the clean picture (slightly lower than your first attempt: broken rows no longer pollute the sample).

**Expected (verified):** **35,336** mismatching rows → **23,057** with a filled `totalRent` (of which **9,641** don't even match `baseRent + serviceCharge`) and **12,279** with a missing `totalRent`.

**Decision (recorded so it isn't lost):** the mismatches **stay in the data**. A total ≠ base + extras is usually *legal* — the listing includes costs the two component columns don't capture (garage, internet, furniture). Dropping ~35k rows (13 %) for arithmetic purity would throw away real listings.

---

### 3c — Apply the sane bounds (the actual drop)

Fixed limits, chosen over 99th-percentile clipping (decision above): **baseRent 10–10,000 € · livingSpace 10–500 m² · noRooms 1–20 · totalRent ≤ 20,000 €**. Missing values are *not* removed here — `totalRent` NaN stays and gets its own decision in Step 4.

**Expected (verified):** bounds remove **216** rows, the totalRent cap removes **8** → Step 3 total: **518 rows removed (0.19 %)**, **268,332 survive (99.81 % of 268,850)**.

In [10]:
# --- 3c: apply the sane bounds (verified expectation: 216 + 8 = 224 removed, 268,332 survive) ---
# what: bound check for baseRent (10-10,000 EUR) — kills the 9,999,999 placeholder prices
# why: 268k rows can't all be hand-checked; fixed bounds remove only provably impossible values
mask_bounds = (
    df_clean['baseRent'].between(10, 10_000)          # rent must exist and be plausible
    & df_clean['livingSpace'].between(10, 500)        # 10 m² studio to 500 m² house
    & df_clean['noRooms'].between(1, 20)              # kills the 999.99-room rows
)

# what: keep only in-bounds rows; count what the bounds alone removed
# why: boolean filtering with a positive mask reads clearer than stacking ~drops
n_bounds_removed = int((~mask_bounds).sum())
df_clean = df_clean[mask_bounds]

# what: cap totalRent at 20,000 EUR, keeping NaN and 0 as-is
# why: NaN + 0 are MISSING values — their fate is a Step 4 decision, so they must survive 3c
total_rent_ok = df_clean['totalRent'].isna() | (df_clean['totalRent'] == 0) | df_clean['totalRent'].between(10, 20_000)
n_cap_removed = int((~total_rent_ok).sum())
df_clean = df_clean[total_rent_ok]

# what: final Step 3 bookkeeping — 3a subtotal vs the bounds/cap removals
# why: the numbers here must match the Step 3 markdown table (518 removed, 268,332 remain)
step3_removed = rows_before - df_clean.shape[0]
print(f"3a removed 294 | bounds removed {n_bounds_removed} | totalRent cap removed {n_cap_removed}")
print(f"Step 3 total removed: {step3_removed} -> {df_clean.shape[0]} rows survive "
      f"({100 * df_clean.shape[0] / 268850:.2f}% of the original 268,850)")

# --- consolidate the name so Steps 4-7 operate on one dataframe called df ---
# what: point df at the cleaned dataframe
# why: the later TODO cells all read `df`; renaming once here avoids carrying two frames around
df = df_clean


3a removed 294 | bounds removed 216 | totalRent cap removed 8
Step 3 total removed: 518 -> 268332 rows survive (99.81% of the original 268,850)


## Step 4 — Handle missing values

**What & why:** Empty cells (`NaN`) remain — 27 of 49 columns have gaps, but they are not equally important. Strategy, decided column group by column group:

1. **Drop columns that are mostly empty (>60 %):** `telekomHybridUploadSpeed` 83.26 %, `electricityBasePrice` 82.57 %, `electricityKwhPrice` 82.57 %, `energyEfficiencyClass` 71.05 %, `lastRefurbish` 69.98 %, `heatingCosts` 68.17 %, `noParkSpaces` 65.39 %. A column that is 2/3 empty carries too little signal to fill honestly — median-filling `heatingCosts` would invent 185k values.
2. **`totalRent` — the big call (decided):** 40,449 rows missing (15.07 %) + 226 zeros. **`baseRent` (0 % missing) becomes the primary price column; `totalRent` stays untouched** — no imputation, no row drops. Where it exists it stays useful (Phase 3 can compare gross vs net rent on the 85 % subset). Filling it from `baseRent + serviceCharge` would systematically understate it (heating costs are about to be dropped), and dropping 15 % of rows would erase good listings.
3. **Fill remaining numeric gaps with the median:** `yearConstructed` 21.18 %, `pricetrend` 0.68 %, `serviceCharge` 2.53 % (+ 3,389 zeros treated as missing → filled too). Median, not mean — the distributions are skewed, the mean is dragged by outliers.
4. **Fill text gaps with `'unknown'`:** `condition` 25.46 %, `heatingType` 16.67 %, `typeOfFlat` 13.61 %, `interiorQual` 41.90 %, `petsAllowed` 42.62 %, etc. `'unknown'` keeps the rows and makes the missingness *visible* in every chart — Tableau will show it as its own category.

**Input:** `df` (post-Step-3: 268,332 rows).

**Hints / verified numbers:**
- Recompute missing % with `df.isna().mean()` — expect the 7-column drop list above.
- After Step 4, *no* column in `df` should have a NaN left except `totalRent` (deliberate) and columns you consciously leave alone.

**Expected output (verified 2026-11-03 on post-Step-3 data):** the 7 columns dropped → 42 remain; after filling, every column except `totalRent` (and any you explicitly exempt) reports 0 missing.

In [11]:
# --- Step 4a: drop columns that are mostly empty (> 60 % missing) ---
# what: compute each column's missing fraction
# why: the >60% threshold is a fixed, documented rule (cleaning_decisions.md §3) — not ad hoc
missing_frac = df.isna().mean()

# what: identify the 7 columns above the 0.60 line
# why: a column that is 2/3 empty carries too little signal to fill honestly — drop, don't invent
drop_cols = missing_frac[missing_frac > 0.60].index.tolist()
print("Dropping:", drop_cols)

# what: drop them and report shape before/after
# why: silent column drops are untraceable; the print is the audit trail
print(f"Columns: {df.shape[1]} -> {df.shape[1] - len(drop_cols)} after dropping mostly-empty columns")
df = df.drop(columns=drop_cols)


Dropping: ['telekomHybridUploadSpeed', 'noParkSpaces', 'heatingCosts', 'energyEfficiencyClass', 'lastRefurbish', 'electricityBasePrice', 'electricityKwhPrice']
Columns: 49 -> 42 after dropping mostly-empty columns


In [12]:
# --- Step 4b: the totalRent decision — recorded as an explicit no-op ---
# what: document that baseRent (0 % missing) is the primary price column and totalRent stays untouched
# why: imputing totals from baseRent + serviceCharge would systematically understate them (heatingCosts is dropped),
#      and dropping the 40,449 rows would erase good listings — decided in docs/cleaning_decisions.md (O1 closed)
print(f"totalRent missing: {df['totalRent'].isna().sum()} rows ({df['totalRent'].isna().mean():.2%}) — kept as-is by decision")
print(f"baseRent missing:  {df['baseRent'].isna().sum()} rows — primary price column for all analyses")

# what: also treat totalRent == 0 as missing-but-kept (226 rows)
# why: 0 is not a legal rent; but like NaN it belongs to the 'no total available' state, not to a fix
print(f"totalRent == 0:    {int((df['totalRent'] == 0).sum())} rows — also kept, flagged as no-total")

# what: recompute which columns still have gaps after the drop
# why: the fill lists in 4c/4d must match the actual remaining columns, not memory
missing_after_drop = df.isna().mean()
print(f"\ncolumns with gaps after drop: {int((missing_after_drop > 0).sum())}")


totalRent missing: 40445 rows (15.07%) — kept as-is by decision
baseRent missing:  0 rows — primary price column for all analyses
totalRent == 0:    226 rows — also kept, flagged as no-total

columns with gaps after drop: 20


In [14]:
# --- Step 4c: fill remaining numeric gaps with the median ---
# what: numeric columns with gaps, verified post-drop (yearConstructed 21.18 %, serviceCharge 2.53 %, pricetrend 0.68 %)
# why: fixed explicit lists beat auto-detecting 'all numeric' — auto-detect could silently touch totalRent, which we chose to leave alone
numeric_fill_cols = ['yearConstructed', 'serviceCharge', 'pricetrend']

# what: loop each column and fill NaN with that column's median
# why: median, not mean — these distributions are skewed; the mean gets dragged by outliers
for col in numeric_fill_cols:
    median_val = df[col].median()
    n_before = int(df[col].isna().sum())
    df[col] = df[col].fillna(median_val)
    print(f"{col}: filled {n_before} missing with median {median_val:,.2f}")

# what: treat serviceCharge == 0 as missing too, then re-fill with the median
# why: a 0 EUR service charge is almost always 'not recorded' rather than genuinely free (3,389 such rows)
mask_zero_sc = df['serviceCharge'] == 0
df.loc[mask_zero_sc, 'serviceCharge'] = df['serviceCharge'].median()
print(f"serviceCharge: {int(mask_zero_sc.sum())} zeros re-filled with median")

# what: fill the 5 numeric columns I deliberately left out of 4c in the first pass
# why: they have gaps but weaker analysis value — floor/numberOfFloors/thermalChar/telekomUploadSpeed get medians, yearConstructedRange a rounded median; keeping them numeric preserves dtype consistency
for col in ['floor', 'numberOfFloors', 'thermalChar', 'telekomUploadSpeed', 'yearConstructedRange']:
    median_val = df[col].median()
    n_before = int(df[col].isna().sum())
    df[col] = df[col].fillna(median_val)
    print(f"{col}: filled {n_before} missing with median {median_val:,.2f}")

# --- Step 4d: fill text gaps with 'unknown' ---
# what: detect text columns robustly — pandas 3.0 gives string columns a new `str` dtype, where `== 'object'` matches nothing
# why: the first run proved `dtype == object` matches nothing under pandas 3.0.2; pandas.api.types handles both old and new dtypes
import pandas.api.types as ptypes

def is_text(series):
    # what: a column counts as text if pandas classifies it as string, or it's the classic object dtype
    # why: covers pandas 2.x (`object`) and pandas 3.x (`str`) so the notebook survives a version change
    return ptypes.is_string_dtype(series) or series.dtype == object

text_fill_cols = [c for c in df.columns if df[c].isna().any() and is_text(df[c])]

# what: fill each text column's NaN with the literal 'unknown' and report
# why: keeps every row while making missingness visible as its own category in Phase 3 charts and Tableau
for col in text_fill_cols:
    n_before = int(df[col].isna().sum())
    df[col] = df[col].fillna('unknown')
    print(f"{col}: filled {n_before} with 'unknown'")

# what: final Step 4 audit — remaining NaN per column
# why: the expected output says ONLY totalRent keeps NaN; anything else means a fill list missed a column
remaining = df.isna().sum()
print("\nRemaining NaN per column (only totalRent should appear):")
print(remaining[remaining > 0].to_string() if (remaining > 0).any() else "none")


yearConstructed: filled 0 missing with median 1,972.00
serviceCharge: filled 0 missing with median 135.00
pricetrend: filled 0 missing with median 3.39
serviceCharge: 0 zeros re-filled with median
floor: filled 51133 missing with median 2.00
numberOfFloors: filled 97507 missing with median 3.00
thermalChar: filled 106203 missing with median 107.00
telekomUploadSpeed: filled 33281 missing with median 40.00
yearConstructedRange: filled 56838 missing with median 3.00
heatingType: filled 44724 with 'unknown'
telekomTvOffer: filled 32542 with 'unknown'
firingTypes: filled 56777 with 'unknown'
houseNumber: filled 70799 with 'unknown'
condition: filled 68328 with 'unknown'
interiorQual: filled 112432 with 'unknown'
petsAllowed: filled 114351 with 'unknown'
streetPlain: filled 70794 with 'unknown'
typeOfFlat: filled 36522 with 'unknown'
description: filled 19705 with 'unknown'
facilities: filled 52777 with 'unknown'

Remaining NaN per column (only totalRent should appear):
totalRent    40445


## Step 5 — Fix dates and location names

**What & why:** (a) The `date` column is text like `Feb20` — it should become a real date. (b) Location names use `_` instead of spaces (e.g. `Nordrhein_Westfalen`), so we build readable `state` and `city` columns.

**Input:** `df`.

**⚠️ Corrected by Phase 1:** `regio2` is **not** a city — it is the district (Kreis) and an exact duplicate of `geo_krs`. The city / neighbourhood column is **`regio3`** (~8,700 unique values, e.g. `Schüren`, `Äußere_Neustadt_Antonstadt`).

**Hints (not the answer):**
- Parse dates with `pd.to_datetime(df['date'], format='%b%y')` — look up what `%b` and `%y` mean. The values are English month abbreviations, which is exactly what `%b` expects.
- You should end up with exactly **4** distinct dates: Sep 2018, May 2019, Oct 2019, Feb 2020. If you get more, something went wrong.
- Clean text with `.str.replace('_', ' ')` and `.str.strip()`.
- Build `state` from `regio1` and `city` from `regio3`. Drop the redundant twins (`geo_bln`, and `geo_krs` *or* `regio2`) while you are here.
- 💡 Note: `regio3` is very granular — most places have only a few dozen listings. Decide later whether to keep it as-is, or additionally group to the district level for analysis.

**Expected output:** `date` becomes datetime with 4 unique values; new `state` and `city` columns show readable names with spaces instead of underscores.

In [ ]:
# TODO 5 — convert the date column; create clean state (from regio1) and city (from regio3) columns

## Step 6 — Feature engineering

**What & why:** Create the columns that make analysis easy: **price per m²** (rent ÷ area — the standard way to compare flats) and **size/room categories** so we can group flats later.

**Input:** `df`.

**Hints (not the answer):**
- `df['price_per_m2'] = df['baseRent'] / df['livingSpace']` — round it, and maybe clip to sane bounds (e.g. 5–100).
- Use `pd.cut(df['livingSpace'], bins=[...], labels=[...])` to make categories like `small`/`medium`/`large`. Pick your own bins that make sense.
- Do the same for rooms, e.g. a `room_category`.

**Expected output:** new columns appear; `df.shape` shows more columns than before.

In [ ]:
# TODO 6 — create price_per_m2, size_category and room_category

## Step 7 — Final cleanup and save

**What & why:** Drop any remaining rows missing critical info (`baseRent`, `livingSpace`, `noRooms`, `state`), reset the index, and save the clean file where Phase 3 will read it.

**Input:** `df`.

**Hints (not the answer):**
- `df.dropna(subset=[...])` removes rows with empty critical cells.
- `df.reset_index(drop=True)` gives a clean 0…N index.
- Save with `df.to_csv('../data/processed/immo_data_clean.csv', index=False)`.
- Print final shape and how much of the original data survived (row-loss %).

**Expected output:** a saved file, a final shape (e.g. ~250k rows) and a row-loss message.

In [ ]:
# TODO 7 — drop rows missing critical columns, reset index, save clean CSV, report final shape

## ✍️ Step 8 — Check your work

Edit this cell and answer honestly:

- [ ] Final row count ≈ what I expected? (write it: ______)
- [ ] `data/processed/immo_data_clean.csv` exists in the file explorer?
- [ ] No obvious `NaN` left in the columns we need?
- [ ] I can explain, in one sentence, what `price_per_m2` means? (write it: ______)

**Anything that confused you (notes for me):**

- 